# SFT Training — Acervo S1 Extractor

Fine-tunes Qwen3.5-9B with LoRA on the S1 extraction dataset.

**Prerequisite:** Run `00_setup/install_deps.ipynb` first.

## 1. Load Model + Apply LoRA

In [4]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3.5-9B"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

free, total = torch.cuda.mem_get_info(0)
print(f"VRAM after LoRA: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")

==((====))==  Unsloth 2026.3.10: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260323+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

## 2. Load Dataset

In [ ]:
from datasets import load_dataset
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
TRAIN_FILE = str(PROJECT_ROOT / "training_data" / "s1_extraction.jsonl")
VAL_FILE = str(PROJECT_ROOT / "training_data" / "s1_validation.jsonl")

# Fallback: if running from project root
if not Path(TRAIN_FILE).exists():
    TRAIN_FILE = "training_data/s1_extraction.jsonl"
    VAL_FILE = "training_data/s1_validation.jsonl"

print(f"Loading from: {TRAIN_FILE}")

dataset = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "validation": VAL_FILE,
})

print(f"Train: {len(dataset['train'])} examples")
print(f"Val:   {len(dataset['validation'])} examples")
print(f"\nSample keys: {list(dataset['train'][0].keys())}")
print(f"Messages in first example: {len(dataset['train'][0]['messages'])}")

Loading from: d:\Development\acervo-graph-model\training_data\s1_extraction.jsonl


Generating train split: 450 examples [00:00, 36712.70 examples/s]
Generating validation split: 50 examples [00:00, 4996.19 examples/s]

Train: 450 examples
Val:   50 examples

Sample keys: ['messages']
Messages in first example: 3


## 3. Format for Training

Apply the chat template to convert messages into the format Qwen expects.

We use `enable_thinking=False` to keep the model in non-thinking mode (no `<think>` tokens).

In [6]:
def format_example(example):
    """Apply chat template to a single example."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

# num_proc=None to avoid subprocess (tokenizer not picklable on Windows)
train_dataset = dataset["train"].map(format_example)
val_dataset = dataset["validation"].map(format_example)

print("--- Sample formatted text (first 500 chars) ---")
print(train_dataset[0]["text"][:500])

Map: 100%|██████████| 50/50 [00:00<00:00, 1183.60 examples/s]

--- Sample formatted text (first 500 chars) ---
<|im_start|>system
You are a knowledge extractor for a personal knowledge graph. Analyze the conversation and return a single JSON object with topic classification, entities, relations, and facts. Output valid JSON only, no markdown, no explanation.<|im_end|>
<|im_start|>user
EXISTING NODES:
[{"id": "vertex_solutions", "label": "Vertex Solutions", "type": "organization", "layer": "PERSONAL", "attributes": {}}, {"id": "skyline_tech", "label": "Skyline Tech", "type": "organization", "layer": "PERS


## 4. Training

In [8]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir="./outputs/s1_sft",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        optim="adamw_8bit",
        seed=42,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        dataset_num_proc=None,  # disable multiprocessing on Windows
        dataloader_num_workers=0,  # no worker subprocesses
        report_to="none",
    ),
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")

Unsloth: Tokenizing ["text"]: 100%|██████████| 50/50 [00:00<00:00, 747.59 examples/s]

Trainable params: 29,097,984
Total params:     5,774,124,272


In [9]:
# Train!
stats = trainer.train()
print(f"\nTraining complete.")
print(f"  Total steps: {stats.global_step}")
print(f"  Train loss:  {stats.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,097,984 of 9,438,911,728 (0.31% trained)


Unsloth: Will smartly offload gradients to save VRAM!


c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with 

Step,Training Loss,Validation Loss
50,0.160616,0.163282
100,0.109612,0.124930
150,0.080218,0.116313


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Training complete.
  Total steps: 171
  Train loss:  0.2243


## 5. Quick Test

In [11]:
import json

# Switch to inference mode
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are a knowledge extractor for a personal knowledge graph. Analyze the conversation and return a single JSON object with topic classification, entities, relations, and facts. Output valid JSON only, no markdown, no explanation."},
    {"role": "user", "content": "EXISTING NODES:\n[]\n\nTOPIC HINT: unresolved — classify the topic yourself\nCURRENT TOPIC: null\n\nPREVIOUS ASSISTANT: null\nUSER: Estoy trabajando en un proyecto llamado Atlas con React y FastAPI. Usamos PostgreSQL."}
]

# Use the underlying tokenizer to avoid multimodal processor issues
inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
text = inner_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)

response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("--- Model Output ---")
print(response)

# Check if it's valid JSON
try:
    parsed = json.loads(response)
    print("\n--- Parsed OK ---")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"\nJSON parse error: {e}")

c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with 

--- Model Output ---
{"topic": {"action": "changed", "label": "Atlas development"}, "entities": [{"id": "atlas", "label": "Atlas", "type": "project", "layer": "PERSONAL", "attributes": {}, "facts": [], "existing_id": null}, {"id": "react", "label": "React", "type": "technology", "layer": "UNIVERSAL", "attributes": {}, "facts": [], "existing_id": null}, {"id": "fastapi", "label": "FastAPI", "type": "technology", "layer": "UNIVERSAL", "attributes": {}, "facts": [], "existing_id": null}, {"id": "postgresql", "label": "PostgreSQL", "type": "technology", "layer": "UNIVERSAL", "attributes": {}, "facts": [], "existing_id": null}, {"id": "luca_rossi", "label": "Luca Rossi", "type": "person", "layer": "PERSONAL", "attributes": {}, "facts": [], "existing_id": null}], "relations": [{"source": "atlas", "target": "react", "relation": "uses_technology"}, {"source": "atlas", "target": "fastapi", "relation": "uses_technology"}, {"source": "atlas", "target": "postgresql", "relation": "uses_technology"}

## 5.5 Stress Test

Runs 20 hand-curated edge cases: empty extractions, dedup, corrections, topic changes, complex literature extraction, mixed Spanish/English.

In [15]:
import json
from pathlib import Path

# Load stress test examples
stress_path = Path.cwd().parent / "training_data" / "s1_stress_test.jsonl"
if not stress_path.exists():
    stress_path = Path("training_data/s1_stress_test.jsonl")

with open(stress_path, encoding="utf-8") as f:
    stress_examples = [json.loads(line) for line in f]

print(f"Running {len(stress_examples)} stress test cases...\n")

inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
FastLanguageModel.for_inference(model)

results = {"pass": 0, "json_fail": 0, "wrong": 0}
failures = []

for i, example in enumerate(stress_examples):
    messages = example["messages"][:2]  # system + user only
    expected = json.loads(example["messages"][2]["content"])

    text = inner_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
    )
    response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    # Check JSON validity
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError:
        results["json_fail"] += 1
        failures.append((i + 1, "JSON_FAIL", response[:100]))
        continue

    # Check key fields
    ok = True
    issues = []

    # Check topic action
    expected_action = expected["topic"]["action"] if isinstance(expected["topic"], dict) else expected["topic"].get("action")
    got_action = parsed.get("topic", {}).get("action")
    if got_action != expected_action:
        issues.append(f"topic: expected={expected_action}, got={got_action}")
        ok = False

    # Check entity count direction (empty vs non-empty)
    expected_has_entities = len(expected.get("entities", [])) > 0
    got_has_entities = len(parsed.get("entities", [])) > 0
    if expected_has_entities != got_has_entities:
        issues.append(f"entities: expected {'non-empty' if expected_has_entities else 'empty'}, got {'non-empty' if got_has_entities else 'empty'}")
        ok = False

    # Check facts count direction
    expected_has_facts = len(expected.get("facts", [])) > 0
    got_has_facts = len(parsed.get("facts", [])) > 0
    if expected_has_facts != got_has_facts:
        issues.append(f"facts: expected {'non-empty' if expected_has_facts else 'empty'}, got {'non-empty' if got_has_facts else 'empty'}")
        ok = False

    if ok:
        results["pass"] += 1
    else:
        results["wrong"] += 1
        # Extract user msg snippet for context
        user_msg = messages[1]["content"].split("USER: ")[-1][:60]
        failures.append((i + 1, "; ".join(issues), user_msg))

total = len(stress_examples)
print(f"Results: {results['pass']}/{total} pass, {results['json_fail']} JSON failures, {results['wrong']} wrong output")
print(f"JSON parse rate: {(total - results['json_fail'])/total:.0%}")
print(f"Accuracy: {results['pass']/total:.0%}")

if failures:
    print(f"\n--- Failures ---")
    for num, issue, ctx in failures:
        print(f"  #{num}: {issue}")
        print(f"       {ctx}")


Running 20 stress test cases...

Results: 9/20 pass, 1 JSON failures, 10 wrong output
JSON parse rate: 95%
Accuracy: 45%

--- Failures ---
  #6: entities: expected empty, got non-empty
       El proyecto de Alice tuvo un problema en producción, se cayó
  #7: entities: expected empty, got non-empty; facts: expected non-empty, got empty
       Mirá, al final decidimos migrar de React a Vue. Ya empezamos
  #9: entities: expected empty, got non-empty
       Nuestro proyecto va bien, el de la empresa. Tiene ya 50 mil 
  #10: facts: expected non-empty, got empty
       No, la app mobile es Compass, no Beacon. Beacon es solo web.
  #11: entities: expected empty, got non-empty
       Estuve leyendo sobre Angular y Vue, parecen interesantes. Pe
  #14: entities: expected empty, got non-empty; facts: expected non-empty, got empty
       Queremos agregar caching a la app principal. Vamos a integra
  #15: topic: expected=subtopic, got=same; facts: expected non-empty, got empty
       Hicimos el spr

## 6. Save Model

In [12]:
# Save LoRA adapter
model.save_pretrained("./outputs/s1_sft/final_lora")
tokenizer.save_pretrained("./outputs/s1_sft/final_lora")
print("LoRA adapter saved to ./outputs/s1_sft/final_lora")

LoRA adapter saved to ./outputs/s1_sft/final_lora


In [14]:
# Export to GGUF for LM Studio
# If running after kernel restart, load the LoRA first:
#   model, tokenizer = FastLanguageModel.from_pretrained("./outputs/s1_sft/final_lora", ...)

model.save_pretrained_gguf(
    "./outputs/s1_sft/gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported to ./outputs/s1_sft/gguf")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: C:\Users\sandy\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 75.49it/s]


Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `./outputs/s1_sft/gguf`: 100%|██████████| 4/4 [00:13<00:00,  3.39s/it]


Successfully copied all 4 files from cache to `./outputs/s1_sft/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:33<00:00,  8.43s/it]


Unsloth: Merge process complete. Saved to `d:\Development\acervo-graph-model\02_training\outputs\s1_sft\gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: llama.cpp folder exists but binaries not found - will build
Unsloth: Missing packages: cmake
Unsloth: Will attempt to install missing system packages.
Unsloth: Installing cmake via winget (Kitware.CMake)...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to install Kitware.CMake via winget.
Install manually: winget install Kitware.CMake